# Data Cleansing Part 1 - TODO 20: Use .at[] to overwrite the bad value at ...

**Week:** 4  
**Source:** `dataCleansing1_practice.ipynb`

## TODO 20  
*Section: Price column*

Use `.at[]` to overwrite the bad value at index `18903` in `price_cr` with the mean you just computed.

### Prerequisites
Runs the setup and the earlier steps this task depends on (output hidden).

In [ ]:
%%capture
import numpy as np
import pandas as pd
import re

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

df = pd.read_csv("zameenData.csv", index_col=0)
df = df.rename(columns={"pupose": "purpose"})   # fix the typo in the column name
df.head()

df = df.drop_duplicates()

df = df.drop(columns=["agent", "name"])

wrong_scrap = df[~df["purpose"].isin(["For Rent", "For Sale"])]
wrong_scrap

df = df.drop(index=wrong_scrap.index)

df = df[df["purpose"] == "For Sale"]
df.shape

print(df["price"].value_counts().head())
print(df.loc[18903, "price"])      # the bad/garbage value

def convert_into_crore(text):
    """Convert strings like 'PKR1.2 Crore' into a number of Crore."""
    if text is None or not isinstance(text, str):
        return None
    unit = re.search(r"PKR\d+\.?\d*\s*([a-zA-Z]+)", text)
    amount = re.search(r"PKR(\d+\.?\d*)", text)
    if not unit or not amount:
        return None
    value = float(amount.group(1))
    unit = unit.group(1).lower()
    if unit == "lakh":
        return value / 100          # 1 Crore = 100 Lakh
    if unit == "crore":
        return value
    if unit == "arab":
        return value * 100          # 1 Arab = 100 Crore
    return None

df.insert(2, "price_cr", df["price"].apply(convert_into_crore))

same_listings = df[(df["location"] == "LDA Road, Lahore, Punjab")
                   & (df["land_type"] == "Residential Plot")
                   & (df["area"] == "5 Marla")]
mean_price = round(same_listings["price_cr"].mean(), 2)
print("Comparable listings:", len(same_listings), "| mean price_cr:", mean_price)

### Solution

In [ ]:
df.at[18903, "price_cr"] = mean_price
df.loc[18903, ["price", "price_cr"]]